In [ ]:
%load_ext watermark


In [ ]:
import itertools as it
import os
import string

import matplotlib as mpl
from matplotlib import pyplot as plt
import numpy as np
import pandas as pd
from phyloframe import _auxlib as pfa
from phyloframe import legacy as pfl
from pyfonts import load_google_font
import seaborn as sns
from teeplot import teeplot as tp

import pylib  # noqa: F401


In [ ]:
%watermark -diwmuv -iv


In [ ]:
teeplot_subdir = os.environ.get("NOTEBOOK_NAME", "2026-03-04-phylometrics")
teeplot_subdir


In [ ]:
pfa.seed_random(1)


In [ ]:
font = load_google_font("Merriweather", weight=300)
mpl.font_manager.fontManager.addfont(font.get_file())
plt.rcParams["font.family"] = font.get_name()


## Prep Data


In [ ]:
df_pure = pfl.alifestd_join_roots(
    pylib.read_parquet_with_retry("https://osf.io/download/pfvsg/"),
)
df_pure


In [ ]:
df_sweep = pfl.alifestd_join_roots(
    pylib.read_parquet_with_retry("https://osf.io/download/nk69s/"),
)
df_sweep


In [ ]:
dfs = []
for df in (df_pure, df_sweep):
    df["x"] = df["position"] // df["nCol"]
    df["x_"] = df["x"] / df["nRow"]
    df["y"] = df["position"] % df["nCol"]
    df["y_"] = df["y"] / df["nCol"]

    df["origin_time"] = df["dstream_rank"]
    df = pfl.alifestd_mark_origin_time_delta_asexual(df, mutate=True)
    df["log_origin_time_delta"] = np.log10(df["origin_time_delta"] + 1)
    df["branch_length"] = df["origin_time_delta"]
    df["log_branch_length"] = df["log_origin_time_delta"]

    df = pfl.alifestd_mark_sackin_index_asexual(df, mutate=True)
    df = pfl.alifestd_mark_colless_like_index_mdm_asexual(df, mutate=True)
    df = pfl.alifestd_mark_leaves(df, mutate=True)
    df = pfl.alifestd_mark_roots(df, mutate=True)

    df["log_sackin_index"] = np.log10(df["sackin_index"] + 1)
    df["log_colless_like_index_mdm"] = np.log10(
        df["colless_like_index_mdm"] + 1
    )

    df["tree_richness"] = df["log_branch_length"]
    df["tree_depth"] = df["sackin_index"]
    df["tree_imbalance"] = df["colless_like_index_mdm"]

    dfs.append(df)

df_pure, df_sweep = dfs


## Helpers


In [ ]:
def make_xlabel(raw: str) -> str:
    raw = raw.replace("colless_like", "Colless-like")
    raw = raw.removesuffix("_mdm")
    if raw.startswith("log_"):
        name = raw.removeprefix("log_")
        # raw = fr"$\log_{{10}}(\text{{{string.capwords(name.replace('_', ' '))}}})$"
        raw = rf"log₁₀({string.capwords(name.replace('_', ' '))})"
    else:
        raw = string.capwords(raw.replace("_", " "))
    return raw


## Plot Tree Shape


In [ ]:
for regime, layout in it.product(
    ("pure", "sweep"),
    ("vertical", "horizontal", "radial"),
):
    df = {"pure": df_pure, "sweep": df_sweep}[regime]
    edge_color = {"pure": "chocolate", "sweep": "blue"}[regime]

    with tp.teed(
        pylib.tree.draw_tree,
        pfl.alifestd_downsample_tips_asexual(
            df,
            n_downsample=3_000,
            drop_topological_sensitivity=True,
        ),
        edge_alpha=0.5,
        edge_color=edge_color,
        layout=layout,
        margins=-0.05,
        teeplot_outattrs={"regime": regime},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        teed.figure.set_size_inches(4, 2)


## Plot Leaf Tree Stats


In [ ]:
data = pd.concat(
    [
        df_pure[df_pure["is_leaf"]].assign(regime="purifying"),
        df_sweep[df_sweep["is_leaf"]].assign(regime="adaptive"),
    ],
    ignore_index=True,
)


In [ ]:
for x, legend in it.product(
    ("origin_time",),
    (True, False),
):
    with tp.teed(
        sns.kdeplot,
        data=data,
        x=x,
        hue="regime",
        hue_order=["purifying", "adaptive"],
        alpha=0.5,
        common_norm=False,
        fill=True,
        legend=legend,
        linewidth=0,
        palette=["chocolate", "blue"],
        teeplot_outattrs={"legend": legend, "what": "leaf"},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        if legend:
            sns.move_legend(
                teed,
                "lower center",
                bbox_to_anchor=(0.5, 1),
                columnspacing=0.8,
                frameon=False,
                handletextpad=0.3,
                ncol=3,
                title=None,
            )
        sns.despine(ax=teed, left=True)
        teed.set_yticks([])
        teed.figure.set_size_inches(2.5, 0.5)
        teed.set_xlabel(make_xlabel(x))
        teed.set_ylabel("")


In [ ]:
for x, legend in it.product(
    ("origin_time",),
    (True, False),
):
    with tp.teed(
        sns.barplot,
        data=data,
        x=x,
        y="regime",
        hue="regime",
        hue_order=["purifying", "adaptive"],
        alpha=0.5,
        legend=legend,
        palette=["chocolate", "blue"],
        teeplot_outattrs={"legend": legend, "what": "leaf"},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        if legend:
            sns.move_legend(
                teed,
                "lower center",
                bbox_to_anchor=(0.5, 1),
                columnspacing=0.8,
                frameon=False,
                handletextpad=0.3,
                ncol=3,
                title=None,
            )
        sns.despine(ax=teed, left=True)
        teed.set_yticks([])
        teed.figure.set_size_inches(2.5, 0.5)
        teed.set_xlabel(make_xlabel(x))
        teed.set_ylabel("")


## Plot whole-tree stats


In [ ]:
data = pd.concat(
    [
        df_pure.assign(regime="purifying"),
        df_sweep.assign(regime="adaptive"),
    ],
    ignore_index=True,
)


In [ ]:
for x, legend, what in it.product(
    (
        "branch_length",
        "log_branch_length",
        "origin_time",
        "origin_time_delta",
        "log_origin_time_delta",
        "sackin_index",
        "log_sackin_index",
        "colless_like_index_mdm",
        "log_colless_like_index_mdm",
        "tree_depth",
        "tree_imbalance",
        "tree_richness",
    ),
    (True, False),
    ("inner", "all"),
):
    if what == "inner":
        data_ = data[~data["is_leaf"]]
    else:
        data_ = data

    with tp.teed(
        sns.kdeplot,
        data=data_,
        x=x,
        hue="regime",
        hue_order=["purifying", "adaptive"],
        alpha=0.5,
        common_norm=False,
        fill=True,
        legend=legend,
        linewidth=0,
        palette=["chocolate", "blue"],
        teeplot_outattrs={"legend": legend, "what": what},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        if legend:
            sns.move_legend(
                teed,
                "lower center",
                bbox_to_anchor=(0.5, 1),
                columnspacing=0.8,
                frameon=False,
                handletextpad=0.3,
                ncol=3,
                title=None,
            )
        sns.despine(ax=teed, left=True)
        teed.set_yticks([])
        teed.figure.set_size_inches(2.5, 0.5)
        teed.set_xlabel(make_xlabel(x))
        teed.set_ylabel("")
        teed.get_xaxis().get_offset_text().set_position((0.95, 0))
        teed.figure.tight_layout()
        teed.figure.draw_without_rendering()
        teed.text(
            1.0,
            -0.1,
            teed.get_xaxis().get_offset_text().get_text(),
            ha="right",
            va="top",
            transform=teed.transAxes,
        )
        teed.get_xaxis().get_offset_text().set_visible(False)


In [ ]:
for x, legend, what, est in it.product(
    (
        "branch_length",
        "log_branch_length",
        "origin_time",
        "origin_time_delta",
        "log_origin_time_delta",
        "sackin_index",
        "log_sackin_index",
        "colless_like_index_mdm",
        "log_colless_like_index_mdm",
        "tree_depth",
        "tree_imbalance",
        "tree_richness",
    ),
    (True, False),
    ("inner", "all", "root"),
    (np.mean, np.median),
):

    if what == "inner":
        data_ = data[~data["is_leaf"]]
    elif what == "root":
        data_ = data[data["is_root"]]
    elif what == "all":
        data_ = data
    else:
        raise ValueError

    with tp.teed(
        sns.barplot,
        data=data_,
        x=x,
        y="regime",
        hue="regime",
        hue_order=["purifying", "adaptive"],
        alpha=0.5,
        estimator=est,
        legend=legend,
        palette=["chocolate", "blue"],
        teeplot_outattrs={"est": est.__name__, "legend": legend, "what": what},
        teeplot_subdir=teeplot_subdir,
    ) as teed:
        if legend:
            sns.move_legend(
                teed,
                "lower center",
                bbox_to_anchor=(0.5, 1),
                columnspacing=0.8,
                frameon=False,
                handletextpad=0.3,
                ncol=3,
                title=None,
            )
        sns.despine(ax=teed, left=True)
        teed.set_yticks([])
        teed.figure.set_size_inches(2.5, 0.5)
        teed.set_xlabel(make_xlabel(x))
        teed.set_ylabel("")
        teed.figure.tight_layout()
        teed.figure.draw_without_rendering()
        teed.text(
            1.0,
            -0.1,
            teed.get_xaxis().get_offset_text().get_text(),
            ha="right",
            va="top",
            transform=teed.transAxes,
        )
        teed.get_xaxis().get_offset_text().set_visible(False)
